# Phase 3 — Wide & Deep Network for Game Recommendation

Predicts `voted_up` for (user, game) pairs using all available features.

**Wide component** (memorisation): hand-crafted cross features + LOO aggregates → linear layer

**Deep component** (generalisation): NCF embeddings + BERT embeddings + user genre vectors + game features → DNN

Inputs from:
- Phase 0: LOO aggregate features, game static features, BERT description embeddings, user genre vectors
- Phase 1: NCF user/item embeddings (32-dim each)

Baselines: Logistic Regression, XGBoost

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"  # macOS Accelerate framework
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import os
import pickle
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, classification_report,
    precision_recall_curve
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
# Auto-detect base directory (works locally when notebook is in "phase 3/" folder)
from pathlib import Path
_BASE = Path.cwd()  # Jupyter sets cwd to notebook directory

# FEAT_DIR   = "/content/drive/MyDrive/feature_engineered_data"
# NCF_DIR    = "/content/drive/MyDrive/phase1_finalised/phase1_outputs"
# OUTPUT_DIR = "/content/drive/MyDrive/phase3_finalised/without_extra_filter/phase3_outputs"
FEAT_DIR   = str(_BASE.parent.parent / "phase0_finalised" / "feature_engineered_data")
NCF_DIR    = str(_BASE.parent.parent / "phase1_finalised" / "phase1_outputs")
OUTPUT_DIR = str(_BASE / "phase3_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
BATCH_SIZE   = 2048
LR           = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS       = 50
PATIENCE     = 5
DROPOUT_DEEP = 0.3
DROPOUT_COMB = 0.2
VAL_SPLIT    = 0.1
K            = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load Data

In [3]:
# Phase 0 outputs
train_raw = pd.read_parquet(os.path.join(FEAT_DIR, "train_with_features.parquet"))
test_raw  = pd.read_parquet(os.path.join(FEAT_DIR, "test_with_features.parquet"))
neg_raw   = pd.read_parquet(os.path.join(FEAT_DIR, "negatives_with_features.parquet"))

game_features = pd.read_parquet(os.path.join(FEAT_DIR, "game_features_static.parquet"))
game_features["required_age"] = pd.to_numeric(game_features["required_age"], errors="coerce").fillna(0)  # WHY: raw Steam HTML scrape leaves JS strings in this column
bert_emb      = pd.read_parquet(os.path.join(FEAT_DIR, "game_description_embeddings.parquet"))
bert_emb = bert_emb[~bert_emb.index.isna()].groupby(level=0).mean()  # WHY: deduplicate per-appid (avg 30 chunks/game in raw file)
user_genre    = pd.read_parquet(os.path.join(FEAT_DIR, "user_genre_vectors.parquet"))

# Phase 1 outputs
ncf_user_emb = pd.read_parquet(os.path.join(NCF_DIR, "ncf_user_embeddings.parquet"))
ncf_item_emb = pd.read_parquet(os.path.join(NCF_DIR, "ncf_item_embeddings.parquet"))

print(f"Train: {train_raw.shape}, Test: {test_raw.shape}, Neg: {neg_raw.shape}")

print(f"Game features: {game_features.shape}, BERT emb: {bert_emb.shape}")
print(f"User genre: {user_genre.shape}")
print(f"NCF user emb: {ncf_user_emb.shape}, NCF item emb: {ncf_item_emb.shape}")

Train: (173832, 36), Test: (51063, 36), Neg: (690537, 14)
Game features: (32883, 94), BERT emb: (32883, 128)
User genre: (32907, 28)
NCF user emb: (36840, 34), NCF item emb: (32883, 34)


## 2. Feature Assembly

Merge all feature sources into unified feature matrices for train, test, and negatives.

**Shared columns** across train/test/neg: `author_steamid`, `appid`, `voted_up`, 9 LOO features.
Everything else comes from merges on `appid` or `author_steamid`.

In [4]:
# Column definitions
LOO_COLS = [
    "user_positive_ratio", "user_avg_playtime", "user_review_count_loo",
    "user_price_preference", "game_positive_ratio", "game_avg_playtime",
    "game_review_count_loo", "playtime_vs_user_avg", "developer_avg_rating",
]

GENRE_COLS      = [c for c in game_features.columns if c.startswith("genre_")]
CAT_COLS        = [c for c in game_features.columns if c.startswith("cat_")]
PLATFORM_COLS   = [c for c in game_features.columns if c.startswith("platform_")]
GAME_SCALAR_COLS = ["is_free", "required_age", "metacritic_score", "mat_final_price_log", "publisher_portfolio_size"]

BERT_COLS       = [f"desc_emb_{i}" for i in range(128)]
USER_GENRE_COLS = [c for c in user_genre.columns]
NCF_USER_COLS   = [c for c in ncf_user_emb.columns if c.startswith("ncf_user_emb_")]
NCF_ITEM_COLS   = [c for c in ncf_item_emb.columns if c.startswith("ncf_item_emb_")]

print(f"LOO: {len(LOO_COLS)}, Genre: {len(GENRE_COLS)}, Cat: {len(CAT_COLS)}, Platform: {len(PLATFORM_COLS)}")
print(f"Game scalar: {len(GAME_SCALAR_COLS)}, BERT: {len(BERT_COLS)}")
print(f"User genre: {len(USER_GENRE_COLS)}, NCF user: {len(NCF_USER_COLS)}, NCF item: {len(NCF_ITEM_COLS)}")

LOO: 9, Genre: 28, Cat: 58, Platform: 3
Game scalar: 5, BERT: 128
User genre: 28, NCF user: 32, NCF item: 32


In [5]:
def assemble_features(df):
    """Merge all feature sources onto a base DataFrame with (author_steamid, appid, voted_up, LOO features)."""
    base = df[["author_steamid", "appid", "voted_up"] + LOO_COLS].copy()

    # Game static features (genre, category, platform multi-hot + scalars)
    base = base.merge(game_features, left_on="appid", right_index=True, how="left")

    # BERT description embeddings (128-dim)
    base = base.merge(bert_emb, left_on="appid", right_index=True, how="left")

    # User genre preference vectors (28-dim)
    base = base.merge(user_genre, left_on="author_steamid", right_index=True, how="left")

    # NCF user embeddings (32-dim)
    base = base.merge(
        ncf_user_emb[["author_steamid"] + NCF_USER_COLS],
        on="author_steamid", how="left"
    )

    # NCF item embeddings (32-dim)
    base = base.merge(
        ncf_item_emb[["appid"] + NCF_ITEM_COLS],
        on="appid", how="left"
    )

    base = base.fillna(0)
    return base


print("Assembling train features...")
train_feat = assemble_features(train_raw)
print("Assembling test features...")
test_feat = assemble_features(test_raw)
print("Assembling negative features...")
neg_feat = assemble_features(neg_raw)

print(f"\nTrain: {train_feat.shape}, Test: {test_feat.shape}, Neg: {neg_feat.shape}")

Assembling train features...
Assembling test features...
Assembling negative features...

Train: (173832, 326), Test: (51063, 326), Neg: (690537, 326)


## 3. Wide Cross Features

Hand-crafted interaction features for the wide (memorisation) component:
- **genre_match_score**: dot product of user genre preferences × game genre vector
- **price_gap**: absolute difference between user's typical price and game price
- **free_match**: interaction between is_free and user's preference for free games

In [6]:
def add_cross_features(df):
    """Add wide cross-product features in-place."""
    # Genre match: dot product of user genre prefs and game genre vector
    user_g = df[USER_GENRE_COLS].values
    game_g = df[GENRE_COLS].values
    df["genre_match_score"] = np.sum(user_g * game_g, axis=1)

    # Price gap: |user_price_preference - game_price|
    df["price_gap"] = np.abs(df["user_price_preference"].values - df["mat_final_price_log"].values)

    # Free game match: is_free * (fraction of user's free-game reviews approximated from price pref)
    df["free_match"] = df["is_free"].values * (1.0 / (1.0 + df["user_price_preference"].values))

    return df


CROSS_COLS = ["genre_match_score", "price_gap", "free_match"]

for feat_df in [train_feat, test_feat, neg_feat]:
    add_cross_features(feat_df)

print("Cross features added.")
print(f"Sample genre_match_score: {train_feat['genre_match_score'].describe().to_dict()}")

Cross features added.
Sample genre_match_score: {'count': 173832.0, 'mean': 1.6719868712290442, 'std': 1.0468910022388878, 'min': 0.0, '25%': 1.0, '50%': 1.6094416648761474, '75%': 2.1518789831282152, 'max': 10.0}


## 4. Define Feature Groups & Prepare Training Data

In [7]:
# Wide features: LOO aggregates + cross features
WIDE_FEATURES = LOO_COLS + CROSS_COLS

# Deep features: embeddings + game features
DEEP_FEATURES = (
    NCF_USER_COLS + NCF_ITEM_COLS +
    BERT_COLS +
    USER_GENRE_COLS +
    GENRE_COLS + CAT_COLS + PLATFORM_COLS +
    GAME_SCALAR_COLS +
    LOO_COLS
)

ALL_FEATURES = WIDE_FEATURES + [f for f in DEEP_FEATURES if f not in WIDE_FEATURES]

print(f"Wide features: {len(WIDE_FEATURES)}")
print(f"Deep features: {len(DEEP_FEATURES)}")
print(f"Total unique features: {len(ALL_FEATURES)}")

Wide features: 12
Deep features: 323
Total unique features: 326


In [8]:
# Combine train + negatives for model training
combined_train = pd.concat([train_feat, neg_feat], ignore_index=True)
combined_train = combined_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

labels_combined = combined_train["voted_up"].astype(float).values

# Real-reviews-only validation set (for early stopping + classification threshold)
# Split real training reviews into 90% train / 10% val
real_labels = train_feat["voted_up"].astype(float).values
real_train_idx, real_val_idx = train_test_split(
    np.arange(len(train_feat)), test_size=VAL_SPLIT, random_state=SEED,
    stratify=real_labels
)
val_real = train_feat.iloc[real_val_idx].copy()

# Build ranking evaluation set: for each val positive, sample 99 negatives
N_RANK_NEG = 99
rng = np.random.default_rng(SEED)
neg_appids_by_user = neg_feat.groupby("author_steamid")["appid"].apply(set).to_dict()

def build_ranking_set(eval_df, neg_lookup, n_neg=99, rng=None):
    """For each positive in eval_df, sample n_neg negatives for ranking evaluation."""
    pos_df = eval_df[eval_df["voted_up"] == True].copy()
    records = []
    all_neg_appids = neg_feat["appid"].unique()
    for _, row in pos_df.iterrows():
        uid = row["author_steamid"]
        pos_appid = row["appid"]
        # Get this user's negative pool
        user_negs = neg_lookup.get(uid, set())
        user_negs = user_negs - {pos_appid}
        if len(user_negs) < 1:
            continue
        n_sample = min(n_neg, len(user_negs))
        sampled = rng.choice(list(user_negs), size=n_sample, replace=False)
        records.append({"author_steamid": uid, "appid": pos_appid, "label": 1.0})
        for neg_app in sampled:
            records.append({"author_steamid": uid, "appid": neg_app, "label": 0.0})
    return pd.DataFrame(records)

val_rank_df = build_ranking_set(val_real, neg_appids_by_user, n_neg=N_RANK_NEG, rng=rng)

print(f"Combined training rows: {len(combined_train):,}")
print(f"  Positive: {labels_combined.sum():,.0f} ({labels_combined.mean():.1%})")
print(f"  Negative: {(1-labels_combined).sum():,.0f} ({1-labels_combined.mean():.1%})")
print(f"\nReal-review validation: {len(val_real):,} (pos {val_real['voted_up'].mean():.1%})")
print(f"Ranking validation: {len(val_rank_df):,} records ({(val_rank_df['label']==1).sum():,} pos + {(val_rank_df['label']==0).sum():,} neg)")
print(f"Test rows: {len(test_feat):,}")


Combined training rows: 864,369
  Positive: 129,846 (15.0%)
  Negative: 734,523 (85.0%)

Real-review validation: 17,384 (pos 74.7%)
Ranking validation: 577,191 records (12,984 pos + 564,207 neg)
Test rows: 51,063


In [9]:
# Training: use ALL combined data (real + synthetic negatives)
# Validation for early stopping: real reviews only

scaler = StandardScaler()

# Training arrays (full combined)
X_train_wide = scaler.fit_transform(combined_train[WIDE_FEATURES].values)
X_train_deep = combined_train[DEEP_FEATURES].values.astype(np.float32)
y_train = labels_combined

# Validation arrays (real reviews only — for early stopping)
X_val_wide = scaler.transform(val_real[WIDE_FEATURES].values)
X_val_deep = val_real[DEEP_FEATURES].values.astype(np.float32)
y_val = val_real["voted_up"].astype(float).values

# Test arrays (real reviews only — for classification metrics)
X_test_wide = scaler.transform(test_feat[WIDE_FEATURES].values)
X_test_deep = test_feat[DEEP_FEATURES].values.astype(np.float32)
y_test = test_feat["voted_up"].astype(float).values

print(f"Train: {len(y_train):,} (pos {y_train.mean():.1%}) — includes synthetic negatives")
print(f"Val:   {len(y_val):,} (pos {y_val.mean():.1%}) — real reviews only")
print(f"Test:  {len(y_test):,} (pos {y_test.mean():.1%}) — real reviews only")
print(f"Wide dim: {X_train_wide.shape[1]}, Deep dim: {X_train_deep.shape[1]}")


Train: 864,369 (pos 15.0%) — includes synthetic negatives
Val:   17,384 (pos 74.7%) — real reviews only
Test:  51,063 (pos 76.1%) — real reviews only
Wide dim: 12, Deep dim: 323


## 5. PyTorch Dataset & DataLoaders

In [10]:
class WideDeepDataset(Dataset):
    def __init__(self, X_wide, X_deep, y):
        self.X_wide = torch.tensor(X_wide, dtype=torch.float32)
        self.X_deep = torch.tensor(X_deep, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_wide[idx], self.X_deep[idx], self.y[idx]


train_dataset = WideDeepDataset(X_train_wide, X_train_deep, y_train)
val_dataset   = WideDeepDataset(X_val_wide, X_val_deep, y_val)
test_dataset  = WideDeepDataset(X_test_wide, X_test_deep, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE * 4, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 4, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Train batches: 423, Val batches: 3, Test batches: 7


## 6. Wide & Deep Model Architecture

Following Cheng et al. (2016):
- **Wide**: Linear layer on cross features + LOO aggregates
- **Deep**: DNN on dense embeddings and game features
- **Combined**: Concatenate wide + deep outputs → final prediction

In [11]:
class WideAndDeep(nn.Module):
    def __init__(self, wide_dim, deep_dim, deep_layers=(256, 128, 64), dropout_deep=0.3, dropout_comb=0.2):
        super().__init__()

        # Wide component: linear
        self.wide = nn.Linear(wide_dim, 1)

        # Deep component: MLP with batch norm
        layers = []
        in_dim = deep_dim
        for out_dim in deep_layers:
            layers += [
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(dropout_deep),
            ]
            in_dim = out_dim
        self.deep = nn.Sequential(*layers)

        # Combined output: deep last layer + wide output
        self.combined = nn.Sequential(
            nn.Linear(deep_layers[-1] + 1, 32),
            nn.ReLU(),
            nn.Dropout(dropout_comb),
            nn.Linear(32, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x_wide, x_deep):
        wide_out = self.wide(x_wide)          # (B, 1)
        deep_out = self.deep(x_deep)           # (B, 64)
        combined = torch.cat([wide_out, deep_out], dim=-1)  # (B, 65)
        logits = self.combined(combined).squeeze(-1)        # (B,)
        return torch.sigmoid(logits)


model = WideAndDeep(
    wide_dim=len(WIDE_FEATURES),
    deep_dim=len(DEEP_FEATURES),
    deep_layers=[256, 128, 64],
    dropout_deep=DROPOUT_DEEP,
    dropout_comb=DROPOUT_COMB,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

WideAndDeep(
  (wide): Linear(in_features=12, out_features=1, bias=True)
  (deep): Sequential(
    (0): Linear(in_features=323, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
  )
  (combined): Sequential(
    (0): Linear(in_features=65, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters: 127,150


## 7. Training Loop with Early Stopping

In [12]:
# Class weighting based on training distribution
pos_weight = (1 - y_train.mean()) / y_train.mean()
print(f"Positive class weight: {pos_weight:.2f}")

criterion = nn.BCELoss(reduction="none")
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def weighted_bce_loss(preds, targets, pos_weight):
    weights = torch.where(targets == 1, pos_weight, 1.0)
    loss = criterion(preds, targets)
    return (loss * weights).mean()


def predict_scores(model, loader, device):
    """Run inference and return (predictions, labels) arrays."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_w, x_d, y in loader:
            x_w, x_d = x_w.to(device), x_d.to(device)
            preds = model(x_w, x_d).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(y.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)


def score_dataframe(model, feat_df, wide_features, deep_features, scaler, device, batch_size=8192):
    """Score a feature DataFrame and return predictions array."""
    X_w = scaler.transform(feat_df[wide_features].values)
    X_d = feat_df[deep_features].values.astype(np.float32)
    scores = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_w), batch_size):
            w = torch.tensor(X_w[start:start+batch_size], dtype=torch.float32).to(device)
            d = torch.tensor(X_d[start:start+batch_size], dtype=torch.float32).to(device)
            scores.append(model(w, d).cpu().numpy())
    return np.concatenate(scores)


def compute_ranking_metrics_from_df(rank_df, score_col="score", k=10):
    """HR@K and NDCG@K from a ranking DataFrame with (author_steamid, appid, label, score)."""
    hits, ndcgs = [], []
    for uid, group in rank_df.groupby("author_steamid"):
        pos_items = set(group[group["label"] == 1]["appid"])
        if len(pos_items) == 0:
            continue
        ranked = group.sort_values("score", ascending=False)["appid"].values[:k]
        hit = 0
        ndcg = 0.0
        for rank, item in enumerate(ranked, 1):
            if item in pos_items:
                hit = 1
                ndcg = max(ndcg, 1.0 / np.log2(rank + 1))
        hits.append(hit)
        ndcgs.append(ndcg)
    return float(np.mean(hits)), float(np.mean(ndcgs))


Positive class weight: 5.66


In [13]:
best_auc = 0.0
best_epoch = 0
patience_counter = 0
CKPT_PATH = os.path.join(OUTPUT_DIR, "wide_deep_best.pt")

print(f"Training Wide & Deep (up to {EPOCHS} epochs, patience={PATIENCE})")
print(f"Validation on real reviews only (natural distribution)")
print(f"{'Epoch':>6}  {'Loss':>8}  {'Val AUC-ROC':>12}  {'Val AUC-PR':>11}  {'':10}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for x_w, x_d, y in train_loader:
        x_w, x_d, y = x_w.to(DEVICE), x_d.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = weighted_bce_loss(model(x_w, x_d), y, pos_weight)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)

    # Validate on real reviews (natural distribution)
    val_preds, val_labels = predict_scores(model, val_loader, DEVICE)
    auc_roc = roc_auc_score(val_labels, val_preds)
    auc_pr  = average_precision_score(val_labels, val_preds)

    note = ""
    if auc_roc > best_auc:
        best_auc, best_epoch = auc_roc, epoch
        torch.save(model.state_dict(), CKPT_PATH)
        patience_counter = 0
        note = "<-- best"
    else:
        patience_counter += 1
        note = f"patience {patience_counter}/{PATIENCE}"

    print(f"{epoch:>6}  {avg_loss:>8.4f}  {auc_roc:>12.4f}  {auc_pr:>11.4f}  {note}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest Val AUC-ROC: {best_auc:.4f} (epoch {best_epoch})")


Training Wide & Deep (up to 50 epochs, patience=5)
Validation on real reviews only (natural distribution)
 Epoch      Loss   Val AUC-ROC   Val AUC-PR            
     1    0.9998        0.6514       0.8438  <-- best
     2    0.8272        0.6170       0.8309  patience 1/5
     3    0.7665        0.6144       0.8310  patience 2/5
     4    0.7150        0.5809       0.8178  patience 3/5
     5    0.6714        0.6342       0.8389  patience 4/5
     6    0.6283        0.5696       0.8119  patience 5/5

Early stopping at epoch 6.

Best Val AUC-ROC: 0.6514 (epoch 1)


## 8. Test Evaluation

In [14]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

# Classification metrics on real test reviews (natural distribution)
test_preds, test_labels = predict_scores(model, test_loader, DEVICE)

cls_auc_roc = roc_auc_score(test_labels, test_preds)
cls_auc_pr  = average_precision_score(test_labels, test_preds)

# Find optimal threshold on validation set (real reviews), apply to test
val_preds_final, val_labels_final = predict_scores(model, val_loader, DEVICE)
prec, rec, thresholds = precision_recall_curve(val_labels_final, val_preds_final)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
optimal_threshold = thresholds[np.argmax(f1s)] if len(thresholds) > 0 else 0.5

test_preds_binary = (test_preds >= optimal_threshold).astype(int)

print("=" * 60)
print("WIDE & DEEP — CLASSIFICATION (real test reviews)")
print("=" * 60)
print(f"AUC-ROC:   {cls_auc_roc:.4f}")
print(f"AUC-PR:    {cls_auc_pr:.4f}")
print(f"Threshold: {optimal_threshold:.4f} (tuned on val set)")
print(f"F1:        {f1_score(test_labels, test_preds_binary):.4f}")
print(f"Precision: {precision_score(test_labels, test_preds_binary):.4f}")
print(f"Recall:    {recall_score(test_labels, test_preds_binary):.4f}")
print(f"\n{classification_report(test_labels, test_preds_binary, target_names=['Negative', 'Positive'])}")


WIDE & DEEP — CLASSIFICATION (real test reviews)
AUC-ROC:   0.6604
AUC-PR:    0.8457
Threshold: 0.1659 (tuned on val set)
F1:        0.8601
Precision: 0.7710
Recall:    0.9725

              precision    recall  f1-score   support

    Negative       0.48      0.08      0.14     12197
    Positive       0.77      0.97      0.86     38866

    accuracy                           0.76     51063
   macro avg       0.62      0.53      0.50     51063
weighted avg       0.70      0.76      0.69     51063



### HR@K and NDCG@K (Ranking Metrics)

In [15]:
# Ranking evaluation: 1 positive + 99 sampled negatives per user (NCF protocol)
# Build test ranking set
test_rank_df = build_ranking_set(test_feat, neg_appids_by_user, n_neg=N_RANK_NEG, rng=rng)

# Score ranking candidates
# Merge features for ranking candidates
rank_with_feat = test_rank_df.merge(
    neg_feat[["author_steamid", "appid"] + LOO_COLS].drop_duplicates(["author_steamid", "appid"]),
    on=["author_steamid", "appid"], how="left"
)
# For positives, get LOO features from test_feat
pos_mask = rank_with_feat["label"] == 1
pos_loo = test_rank_df[pos_mask][["author_steamid", "appid"]].merge(
    test_feat[["author_steamid", "appid"] + LOO_COLS],
    on=["author_steamid", "appid"], how="left"
)
for col in LOO_COLS:
    rank_with_feat.loc[pos_mask, col] = pos_loo[col].values

# Fill remaining NaN LOO features with 0
rank_with_feat[LOO_COLS] = rank_with_feat[LOO_COLS].fillna(0)

# Assemble full features for ranking candidates
rank_feat = rank_with_feat[["author_steamid", "appid", "label"]].copy()
for col in LOO_COLS:
    rank_feat[col] = rank_with_feat[col].values
rank_feat["voted_up"] = rank_feat["label"].astype(bool)

# Merge game features, BERT, user genre, NCF embeddings
rank_feat = rank_feat.merge(game_features, left_on="appid", right_index=True, how="left")
rank_feat = rank_feat.merge(bert_emb, left_on="appid", right_index=True, how="left")
rank_feat = rank_feat.merge(user_genre, left_on="author_steamid", right_index=True, how="left")
rank_feat = rank_feat.merge(ncf_user_emb[["author_steamid"] + NCF_USER_COLS], on="author_steamid", how="left")
rank_feat = rank_feat.merge(ncf_item_emb[["appid"] + NCF_ITEM_COLS], on="appid", how="left")
rank_feat = rank_feat.fillna(0)
add_cross_features(rank_feat)

# Score
rank_scores = score_dataframe(model, rank_feat, WIDE_FEATURES, DEEP_FEATURES, scaler, DEVICE)
rank_feat["score"] = rank_scores

wd_hr, wd_ndcg = compute_ranking_metrics_from_df(rank_feat, k=K)

print("=" * 60)
print(f"WIDE & DEEP — RANKING (1 pos + {N_RANK_NEG} neg per user)")
print("=" * 60)
print(f"HR@{K}:   {wd_hr:.4f}")
print(f"NDCG@{K}: {wd_ndcg:.4f}")
print(f"Users evaluated: {rank_feat['author_steamid'].nunique():,}")


WIDE & DEEP — RANKING (1 pos + 99 neg per user)
HR@10:   0.8085
NDCG@10: 0.4489
Users evaluated: 28,308


In [16]:
import psutil, os
proc = psutil.Process(os.getpid())
ram_gb = proc.memory_info().rss / 1e9
print(f'Kernel RAM at this point: {ram_gb:.1f} GB')
print(f'combined_train: {combined_train.shape}, {combined_train.memory_usage(deep=True).sum()/1e9:.2f} GB')
print(f'rank_feat: {rank_feat.shape}, {rank_feat.memory_usage(deep=True).sum()/1e9:.2f} GB')
print('--- Variables in scope ---')
for name, val in list(globals().items()):
    try:
        import numpy as np, pandas as pd, torch
        if isinstance(val, (np.ndarray, pd.DataFrame, torch.Tensor)):
            mb = val.nbytes/1e6 if hasattr(val,'nbytes') else val.memory_usage(deep=True).sum()/1e6
            print(f'  {name}: {type(val).__name__} {getattr(val,"shape","?")} = {mb:.0f} MB')
    except: pass


Kernel RAM at this point: 3.2 GB
combined_train: (864369, 329), 1.61 GB
rank_feat: (1391007, 331), 2.60 GB
--- Variables in scope ---
  train_raw: DataFrame (173832, 36) = 166 MB
  test_raw: DataFrame (51063, 36) = 49 MB
  neg_raw: DataFrame (690537, 14) = 68 MB
  game_features: DataFrame (32883, 94) = 25 MB
  bert_emb: DataFrame (32883, 128) = 17 MB
  user_genre: DataFrame (32907, 28) = 8 MB
  ncf_user_emb: DataFrame (36840, 34) = 5 MB
  ncf_item_emb: DataFrame (32883, 34) = 5 MB
  train_feat: DataFrame (173832, 329) = 323 MB
  test_feat: DataFrame (51063, 329) = 95 MB
  neg_feat: DataFrame (690537, 329) = 1282 MB
  feat_df: DataFrame (690537, 329) = 1282 MB
  combined_train: DataFrame (864369, 329) = 1605 MB
  labels_combined: ndarray (864369,) = 7 MB
  real_labels: ndarray (173832,) = 1 MB
  real_train_idx: ndarray (156448,) = 1 MB
  real_val_idx: ndarray (17384,) = 0 MB
  val_real: DataFrame (17384, 329) = 32 MB
  val_rank_df: DataFrame (577191, 3) = 14 MB
  X_train_wide: ndarray (

## 9. Baselines: Logistic Regression & XGBoost

In [17]:
from sklearn.linear_model import LogisticRegression

# Baselines use all features flattened
X_train_all = combined_train[ALL_FEATURES].values.astype(np.float32)
X_test_all  = test_feat[ALL_FEATURES].values.astype(np.float32)

scaler_all = StandardScaler()
X_train_scaled = scaler_all.fit_transform(X_train_all)
X_test_scaled  = scaler_all.transform(X_test_all)

print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000, class_weight="balanced",
    solver="lbfgs",  # WHY: saga uses OpenMP internally -> crashes macOS Jupyter via fork (same issue as XGBoost n_jobs); lbfgs uses scipy (fork-safe)
    random_state=SEED
)
lr_model.fit(X_train_scaled, labels_combined)

# Classification metrics on real test reviews
lr_test_preds = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc_roc = roc_auc_score(y_test, lr_test_preds)
lr_auc_pr  = average_precision_score(y_test, lr_test_preds)

prec, rec, thresholds = precision_recall_curve(y_test, lr_test_preds)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
lr_threshold = thresholds[np.argmax(f1s)] if len(thresholds) > 0 else 0.5
lr_preds_binary = (lr_test_preds >= lr_threshold).astype(int)
lr_f1 = f1_score(y_test, lr_preds_binary)

# Ranking metrics (score the ranking set)
rank_feat_lr = rank_feat.copy()
X_rank_scaled = scaler_all.transform(rank_feat[ALL_FEATURES].values.astype(np.float32))
rank_feat_lr["score"] = lr_model.predict_proba(X_rank_scaled)[:, 1]
lr_hr, lr_ndcg = compute_ranking_metrics_from_df(rank_feat_lr, k=K)

print(f"Logistic Regression Results:")
print(f"  Classification — AUC-ROC: {lr_auc_roc:.4f}, AUC-PR: {lr_auc_pr:.4f}, F1: {lr_f1:.4f}")
print(f"  Ranking — HR@{K}: {lr_hr:.4f}, NDCG@{K}: {lr_ndcg:.4f}")

Training Logistic Regression...
Logistic Regression Results:
  Classification — AUC-ROC: 0.6237, AUC-PR: 0.8249, F1: 0.8646
  Ranking — HR@10: 0.8290, NDCG@10: 0.4744


In [18]:
from xgboost import XGBClassifier

print("Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    eval_metric="auc",
    early_stopping_rounds=10,
    random_state=SEED,
    n_jobs=1,  # WHY: -1 crashes macOS Jupyter via OpenMP fork,
    tree_method="hist",
)

# Use real-review val set for XGBoost early stopping too
X_val_all = val_real[ALL_FEATURES].values.astype(np.float32)
xgb_model.fit(
    X_train_all, labels_combined,
    eval_set=[(X_val_all, y_val)],
    verbose=20,
)

# Classification metrics on real test reviews
xgb_test_preds = xgb_model.predict_proba(X_test_all)[:, 1]
xgb_auc_roc = roc_auc_score(y_test, xgb_test_preds)
xgb_auc_pr  = average_precision_score(y_test, xgb_test_preds)

prec, rec, thresholds = precision_recall_curve(y_test, xgb_test_preds)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
xgb_threshold = thresholds[np.argmax(f1s)] if len(thresholds) > 0 else 0.5
xgb_preds_binary = (xgb_test_preds >= xgb_threshold).astype(int)
xgb_f1 = f1_score(y_test, xgb_preds_binary)

# Ranking metrics
rank_feat_xgb = rank_feat.copy()
X_rank_xgb = rank_feat[ALL_FEATURES].values.astype(np.float32)
rank_feat_xgb["score"] = xgb_model.predict_proba(X_rank_xgb)[:, 1]
xgb_hr, xgb_ndcg = compute_ranking_metrics_from_df(rank_feat_xgb, k=K)

print(f"\nXGBoost Results:")
print(f"  Classification — AUC-ROC: {xgb_auc_roc:.4f}, AUC-PR: {xgb_auc_pr:.4f}, F1: {xgb_f1:.4f}")
print(f"  Ranking — HR@{K}: {xgb_hr:.4f}, NDCG@{K}: {xgb_ndcg:.4f}")


Training XGBoost...
[0]	validation_0-auc:0.83845
[20]	validation_0-auc:0.90783
[40]	validation_0-auc:0.93521
[60]	validation_0-auc:0.94890
[80]	validation_0-auc:0.96083
[100]	validation_0-auc:0.96938
[120]	validation_0-auc:0.97441
[140]	validation_0-auc:0.97823
[160]	validation_0-auc:0.98134
[180]	validation_0-auc:0.98371
[200]	validation_0-auc:0.98575
[220]	validation_0-auc:0.98737
[240]	validation_0-auc:0.98893
[260]	validation_0-auc:0.99018
[280]	validation_0-auc:0.99149
[299]	validation_0-auc:0.99245

XGBoost Results:
  Classification — AUC-ROC: 0.7234, AUC-PR: 0.8962, F1: 0.8644
  Ranking — HR@10: 0.9947, NDCG@10: 0.9553


## 10. Results Comparison

In [19]:
wd_f1 = f1_score(test_labels, test_preds_binary)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "Wide & Deep"],
    "AUC-ROC": [lr_auc_roc, xgb_auc_roc, cls_auc_roc],
    "AUC-PR": [lr_auc_pr, xgb_auc_pr, cls_auc_pr],
    "F1": [lr_f1, xgb_f1, wd_f1],
    f"HR@{K}": [lr_hr, xgb_hr, wd_hr],
    f"NDCG@{K}": [lr_ndcg, xgb_ndcg, wd_ndcg],
})

print("=" * 70)
print("PHASE 3 — MODEL COMPARISON")
print("=" * 70)
print("\nClassification metrics on real test reviews (natural distribution):")
print("Ranking metrics via NCF protocol (1 pos + 99 neg per user):")
print()
display(results.style.highlight_max(
    subset=["AUC-ROC", "AUC-PR", "F1", f"HR@{K}", f"NDCG@{K}"],
    color="lightgreen"
))


PHASE 3 — MODEL COMPARISON

Classification metrics on real test reviews (natural distribution):
Ranking metrics via NCF protocol (1 pos + 99 neg per user):



,Model,AUC-ROC,AUC-PR,F1,HR@10,NDCG@10
0,Logistic Regression,0.623697,0.824922,0.864637,0.828953,0.474423
1,XGBoost,0.723425,0.896158,0.864371,0.994666,0.955287
2,Wide & Deep,0.660428,0.845743,0.860140,0.808499,0.448857


## 11. Save Outputs for Phase 4

In [20]:
# Save Wide & Deep scores for Phase 4 hybrid combination
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

train_scores = score_dataframe(model, train_feat, WIDE_FEATURES, DEEP_FEATURES, scaler, DEVICE)
test_scores  = score_dataframe(model, test_feat, WIDE_FEATURES, DEEP_FEATURES, scaler, DEVICE)
neg_scores   = score_dataframe(model, neg_feat, WIDE_FEATURES, DEEP_FEATURES, scaler, DEVICE)

for name, feat_df, scores in [
    ("wd_train_scores", train_feat, train_scores),
    ("wd_test_scores", test_feat, test_scores),
    ("wd_neg_scores", neg_feat, neg_scores),
]:
    out = pd.DataFrame({
        "author_steamid": feat_df["author_steamid"].values,
        "appid": feat_df["appid"].values,
        "voted_up": feat_df["voted_up"].values,
        "wd_score": scores,
    })
    path = os.path.join(OUTPUT_DIR, f"{name}.parquet")
    out.to_parquet(path, index=False)
    print(f"Saved {name}: {out.shape} -> {path}")

# Save model comparison results
results.to_csv(os.path.join(OUTPUT_DIR, "phase3_results.csv"), index=False)

print(f"\nAll Phase 3 outputs saved to {OUTPUT_DIR}/")


Saved wd_train_scores: (173832, 4) -> /Users/tanhung/Documents/projects/BT4222/bt4222_phase3/phase3_finalised/without_extra_filter/phase3_outputs/wd_train_scores.parquet
Saved wd_test_scores: (51063, 4) -> /Users/tanhung/Documents/projects/BT4222/bt4222_phase3/phase3_finalised/without_extra_filter/phase3_outputs/wd_test_scores.parquet
Saved wd_neg_scores: (690537, 4) -> /Users/tanhung/Documents/projects/BT4222/bt4222_phase3/phase3_finalised/without_extra_filter/phase3_outputs/wd_neg_scores.parquet

All Phase 3 outputs saved to /Users/tanhung/Documents/projects/BT4222/bt4222_phase3/phase3_finalised/without_extra_filter/phase3_outputs/
